In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import pandas as pd

# Add the directory containing the library to sys.path
import os
import sys
library_path = os.path.abspath(r'C:\git\foraging-strategies\code')
if library_path not in sys.path:
    sys.path.append(library_path)
    
# from tools_fixed import PatchForager
from tools import PatchForager

In [ ]:
dfs = []
with h5py.File('data/data.h5', 'r') as hf:
    for sim_key in hf.keys():  # e.g. 'simulation_0'
        sim_group = hf[sim_key]
        sim_num = int(sim_key.split('_')[-1])
        
        for strategy_key in sim_group.keys():
            print(f"Processing {sim_key}/{strategy_key}")
            dataset = sim_group[strategy_key]
            data_array = dataset[()]
            columns = dataset.attrs['columns']
            df = pd.DataFrame(data_array, columns=columns)
            df['simulation'] = sim_num
            df['strategy'] = strategy_key
            dfs.append(df)

# Combine all into one big DataFrame
all_data = pd.concat(dfs, ignore_index=True)
all_data.to_csv('data/simulation_data_df.csv', index=False)

In [ ]:
all_data.rename(columns={'rewards_in_patch': 'cumulative_rewards',
                              'time_in_patch':'site_number',
                              'failures_in_patch': 'cumulative_failures',
                              'patch_id': 'patch_label',
                              'patch_entry_time': 'patch_number',
                              'prob_reward': 'reward_probability',
                              'simulation':'session'}, inplace=True)

all_data['patch_number'].interpolate(method='linear', inplace=True)
all_data['patch_number'] = all_data.groupby('session')['patch_number'].apply(
    lambda x: x.ne(x.shift()).cumsum() - 1  # Detect changes and assign numbers
).reset_index(drop=True)

all_data['shift_is_choice'] = np.where(all_data['patch_label'] == -1, 0, 1) # Use the interpatch to recover is_choice
all_data['is_choice'] = all_data['shift_is_choice'].shift(-1)
all_data  = all_data.loc[all_data['patch_label'] != -1]
all_data['is_choice'] = all_data['is_choice'].fillna(0)

In [ ]:
# Get unique mice
max_number = 30 # range
step = 2 # bin size
full_x = np.arange(0, max_number + 1)  # Cumulative Rewards: 0 to 11
full_y = np.arange(0, max_number + 1)  

mice = all_data['strategy'].unique()
n_mice = len(mice)
variable = 'leave'
vmax=0.6
# Grid size (adjust as needed)
cols = 3  # Number of columns in the grid
rows = -(-n_mice // cols)  # ceiling division

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows), constrained_layout=True, sharex=True, sharey=True)
axes = axes.flatten()

for i, mouse in enumerate(mice):
    ax = axes[i]
    mouse_df = all_data[all_data['strategy'] == mouse]

    # Count number of samples per bin
    counts = mouse_df.groupby(['cumulative_rewards', 'consecutive_failures']).is_choice.count().unstack()

    # Compute mean (probability) per bin
    probs = mouse_df.groupby(['cumulative_rewards', 'consecutive_failures']).is_choice.mean().unstack()

    # # Mask out bins with fewer than 5 samples
    # probs[counts < 5] = np.nan

    if variable == 'stop':
        plot = probs
        title = 'Probability of stopping'
    else:
        plot = 1 - probs
        title = 'Probability of leaving'
    plot = plot.astype(float)
    plot = plot.fillna(np.nan)  # or use .fillna(0) if you prefer
        
    last_hm = sns.heatmap(plot, ax=ax, cmap='YlGnBu', cbar=False, vmax=vmax)
    ax.invert_yaxis()
    # ax.set_ylabel('Cumulative Rewards')
    # ax.set_xlabel('Consecutive Failures')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, ha='right')
    ax.set_xticklabels(ax.get_xticks().astype(int))
    ax.set_yticklabels(ax.get_yticks().astype(int))
    
    # Set ticks to be centered in the square
    ax.set_yticks(np.arange(1, max_number, step) + 0.5, minor=False)
    ax.set_xticks(np.arange(0, max_number, step) + 0.5, minor=False)
    ax.set_xticklabels([str(x) for x in np.arange(0, max_number, step)])
    ax.set_yticklabels([str(y) for y in np.arange(1, max_number, step)])
    ax.tick_params(axis='x', labelbottom=True)
    ax.tick_params(axis='y')
    ax.tick_params(labelbottom=True, labelleft=True)

    ax.set_xlim(0, max_number-1)
    ax.set_ylim(1, max_number-1)
    ax.set_title(f'Mouse {mouse}')
    
# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

cbar = fig.colorbar(last_hm.collections[0], ax=axes[:n_mice], orientation='vertical', fraction=0.02, pad=0.04)
cbar.set_label(title)

# plt.savefig(os.path.join(results_path, f'model_grid_strategy_x_consecfailures_y_cumrewards_hue_{variable}_.pdf'), bbox_inches='tight')